# RD validity checks (vote margin)
McCrary-style density and covariate balance checks with local-randomization windows.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import ANALYSIS_DIR, PAPER_FIGURES_DIR, PAPER_LOGS_DIR, PAPER_TABLES_DIR
from src.rd import density_discontinuity, select_bandwidth
from src.rd_localrand import select_window_by_balance
from src.viz_style import savefig, set_style

In [2]:
set_style()

pos_path = ANALYSIS_DIR / "rd_event_panel_pos.parquet"
neg_path = ANALYSIS_DIR / "rd_event_panel_neg.parquet"

for path in [pos_path, neg_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing event panel: {path}")

panel_pos = pd.read_parquet(pos_path)
panel_neg = pd.read_parquet(neg_path)

RUNNING = "running_var_vote"

In [3]:
# Density plots and discontinuity stats

def density_block(frame: pd.DataFrame, label: str) -> dict:
    bandwidth = select_bandwidth(frame[RUNNING], quantile=0.3, max_bw=0.1)
    if pd.isna(bandwidth):
        bandwidth = 0.05

    subset = frame[frame[RUNNING].abs() <= bandwidth]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.hist(subset[RUNNING], bins=40, color="#4C72B0", edgecolor="white")
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(f"Running variable density near cutoff ({label})")
    ax.set_xlabel("Vote share margin (market bloc)")
    ax.set_ylabel("Count")
    savefig(fig, PAPER_FIGURES_DIR / "rd_validity" / f"rd_density_{label}")
    plt.close(fig)

    stats = density_discontinuity(frame[RUNNING], bandwidth=bandwidth)
    stats["sample"] = label
    return stats


density_pos = density_block(panel_pos, "pos")
density_neg = density_block(panel_neg, "neg")

density_df = pd.DataFrame([density_pos, density_neg])
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)
density_path = PAPER_TABLES_DIR / "rd_density_vote_margin.csv"
density_df.to_csv(density_path, index=False)

display(density_df.style.set_caption("Density discontinuity (vote margin)"))

,left_n,right_n,log_diff,se,z_stat,bandwidth,sample
0,16,7,-0.826679,0.453163,-1.824239,0.035019,pos
1,17,23,0.302281,0.319847,0.945081,0.046465,neg


In [4]:
# Balance checks by window
balance_vars = [
    "lag1_log_gdp_pc_const",
    "lag1_trade_open_gdp",
    "lag1_inflation_cpi_ann_pct",
    "lag1_efw_summary",
    "lag1_inv_share_gdp",
]

windows = [0.01, 0.02, 0.03, 0.04, 0.05]

choice_pos, table_pos = select_window_by_balance(
    panel_pos,
    RUNNING,
    balance_vars,
    windows=windows,
    p_threshold=0.15,
    cluster="iso3c",
)

choice_neg, table_neg = select_window_by_balance(
    panel_neg,
    RUNNING,
    balance_vars,
    windows=windows,
    p_threshold=0.15,
    cluster="iso3c",
)

PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)
valid_pos_path = PAPER_TABLES_DIR / "rd_validity_vote_margin_pos.csv"
valid_neg_path = PAPER_TABLES_DIR / "rd_validity_vote_margin_neg.csv"

table_pos.to_csv(valid_pos_path, index=False)
table_neg.to_csv(valid_neg_path, index=False)

display(table_pos.style.set_caption("Balance table (positive shocks)"))
display(table_neg.style.set_caption("Balance table (negative shocks)"))

,covariate,coef,se,pvalue,n_obs,window
0,lag1_log_gdp_pc_const,0.688225,0.066957,0.000000,5,0.010000
1,lag1_trade_open_gdp,19.170659,9.060803,0.034364,5,0.010000
2,lag1_inflation_cpi_ann_pct,0.682746,1.594192,0.668454,5,0.010000
3,lag1_efw_summary,0.235000,0.554479,0.671696,5,0.010000
4,lag1_inv_share_gdp,2.362227,2.303386,0.305106,5,0.010000
5,lag1_log_gdp_pc_const,0.565527,0.278570,0.042346,9,0.020000
6,lag1_trade_open_gdp,14.636120,6.918491,0.034387,9,0.020000
7,lag1_inflation_cpi_ann_pct,-1.365985,2.139992,0.523270,9,0.020000
8,lag1_efw_summary,0.438333,0.347803,0.207564,9,0.020000
9,lag1_inv_share_gdp,2.188197,2.620965,0.403784,9,0.020000


,covariate,coef,se,pvalue,n_obs,window
0,lag1_log_gdp_pc_const,0.255854,0.412235,0.534829,15,0.010000
1,lag1_trade_open_gdp,4.513217,21.581443,0.834351,15,0.010000
2,lag1_inflation_cpi_ann_pct,-2.237094,1.079317,0.038201,15,0.010000
3,lag1_efw_summary,0.324444,0.198804,0.102684,15,0.010000
4,lag1_inv_share_gdp,-0.828354,1.775546,0.640833,15,0.010000
5,lag1_log_gdp_pc_const,-0.112036,0.429597,0.794251,22,0.020000
6,lag1_trade_open_gdp,18.675223,16.997311,0.271892,22,0.020000
7,lag1_inflation_cpi_ann_pct,-1.365368,0.978488,0.162900,22,0.020000
8,lag1_efw_summary,0.037265,0.243447,0.878341,22,0.020000
9,lag1_inv_share_gdp,0.102756,1.488601,0.944967,22,0.020000


In [5]:
# Record window choice
window_choice = {
    "positive": {
        "window": choice_pos.window,
        "p_threshold": choice_pos.p_threshold,
        "windows_tested": choice_pos.windows_tested,
    },
    "negative": {
        "window": choice_neg.window,
        "p_threshold": choice_neg.p_threshold,
        "windows_tested": choice_neg.windows_tested,
    },
}

PAPER_LOGS_DIR.mkdir(parents=True, exist_ok=True)
choice_path = PAPER_LOGS_DIR / "rd_window_choice.json"
choice_path.write_text(json.dumps(window_choice, indent=2))

312

## Interpretation
Density continuity and covariate balance are prerequisites for the local
randomization RD design. The selected window(s) define the main estimation
sample for LP-IV impulse responses.